<a href="https://colab.research.google.com/github/Heba2627/NLP-Project/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Dataset via. Pandas

In [18]:
import pandas as pd
import numpy as np
import regex as re

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}
df = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])

In [32]:
print(df.columns)
answerable_df = df[df["answerable"] == True]
unanswerable_df = df[df["answerable"] == False]

print(len(answerable_df))
print(len(unanswerable_df))
print("% of dataset that is answerable:", len(answerable_df)/len(df)*100, "%")
print("% of dataset that is unanswerable:", len(unanswerable_df)/len(df)*100, "%")

Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang', 'question_len', 'context_len'],
      dtype='object')
13962
1381
% of dataset that is answerable: 90.99915270807534 %
% of dataset that is unanswerable: 9.000847291924655 %


In [33]:
#Check for letters in all languages + Mark for arabic
word_pattern = re.compile(r"[\p{L}\p{M}]+", flags=re.UNICODE)

def n_tokens(text):
    return len(word_pattern.findall(str(text)))
def median_iqr(s):
    q1, median, q3 = np.percentile(s, [25, 50, 75])
    return pd.Series({"median": median, "Q1": q1, "Q3": q3, "IQR": q3 - q1})

df["question_len"] = df["question"].apply(n_tokens)
df["context_len"] = df["context"].apply(n_tokens)

#Approach idea: build a mini DataFrame to obtain a table like described in the project pdf.
length_stats = pd.DataFrame({"question_length": median_iqr(df["question_len"]), "context_length": median_iqr(df["context_len"])})
length_stats

,question_length,context_length
median,5.0,84.0
Q1,4.0,56.0
Q3,7.0,122.0
IQR,3.0,66.0


In [37]:
#missing values
missing = df.isna().sum().to_frame("n_missing")
missing["pct_missing"] = (100 * missing["n_missing"]/len(df))
missing

#duplicates
dup_filter = df.duplicated(subset=["question", "context"], keep=False)
dup_amount = dup_filter.sum()
dup_groups = df[dup_filter].groupby(["question","context"]).ngroups
redudant_rows = df.duplicated(subset=["question", "context"], keep="first").sum()
print(dup_groups)
print(dup_amount)
print(redudant_rows)

17
34
17


# Tokenisation

In [45]:
!pip install deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00


In [51]:
from transformers import AutoTokenizer
from deep_translator import GoogleTranslator
from collections import Counter, defaultdict

tok = AutoTokenizer.from_pretrained("xlm-roberta-base")

#We need to root out symbols so check for letter
has_letter = re.compile(r"\p{L}", flags=re.UNICODE)

#NOTE: We will probably need a special case for japanese since its the only
#language that doesnt have whitespace or punctuation!
def tokenize_words_xlmr(text):
    pieces = tok.tokenize(str(text))
    words, current = [], ""
    for p in pieces:
        if p.startswith("▁"):
            if current != "":
                words.append(current)
            current = p[1:]
        else:
            current += p
    if current != "":
        words.append(current)
    return [w.lower() for w in words if w != "" and has_letter.search(w)]

def translate(token, lang):
    try:
      return GoogleTranslator(source=lang, target="en").translate(token)
    except:
      return None

counters = defaultdict(Counter)
for lang, question in zip(df["lang"], df["question"]):
    counters[lang].update(tokenize_words_xlmr(question))

rows = [
    {"lang": lang, "token": token, "count": count}
    for lang, counter in counters.items()
    for token, count in counter.most_common(5)
]
top5_df = pd.DataFrame(rows)
top5_df["In English"] = [translate(token, lang) for token, lang in zip(top5_df["token"], top5_df["lang"])]
top5_df = top5_df[["lang", "token", "In English", "count"]]
top5_df

,lang,token,In English,count
0,bn,হয়,is,467
1,bn,প্রথম,the first,458
2,bn,কী,what,439
3,bn,কে,who,415
4,bn,কত,how much,392
5,ja,pygが最初に発表した曲は何,What was the first song released by pyg?,3
6,ja,オーロラ号の漂流時、オーロラ号に乗組員は何人いた?,How many people were on board the Aurora when ...,3
7,ja,カンザスシティが設立されたのはいつ,When was Kansas City founded?,3
8,ja,クロイツ群が初めて発見されたのはいつ,When was the Kreutz group first discovered?,3
9,ja,ライド,ride,2
